# Teman Kopi — DECAFIA (CoffeeLeaf-CO) → Keras MobileNetV3Small → TF.js

Run cells **top to bottom** on Google Colab (GPU: Runtime → Change runtime type → GPU).

## Before you start — put these files in Colab

| Required | Exact path |
| --- | --- |
| DECAFIA zip | `/content/decafia.zip` |
| Scripts | `/content/training/prepare_decafia.py` |
| | `/content/training/train.py` |
| | `/content/training/export_tfjs.py` |

Upload via the left **Files** panel. This notebook does **not** download the dataset.
If a required file is missing, cells raise `FileNotFoundError`.

Zenodo: https://doi.org/10.5281/zenodo.19931903

Class remap: empty/`SANAS` → `healthy`; `minador`/`coco` → `pest_like`; `roya` → `disease_like`.

## 1. GPU check

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
if not gpus:
    print("WARNING: No GPU detected. Runtime → Change runtime type → GPU, then Restart session.")
else:
    print("GPU OK — TensorFlow will use it automatically.")

## 2. Install dependencies

Pins `numpy<2` so `tensorflowjs` does not crash on modern Colab (`np.object` removed).
This cell can take several minutes. If Colab asks to **Restart runtime**, do that, then continue from section 3 (re-upload files if `/content` was cleared).

In [ ]:
# May take a few minutes — progress is quiet (-q). Wait until "Install done."
!pip -q install "numpy<2" "tensorflow>=2.15,<2.20" "tensorflowjs==4.22.0" scikit-learn

import numpy as np
import tensorflow as tf
import tensorflowjs as tfjs

print("Install done.")
print("numpy", np.__version__)
print("tensorflow", tf.__version__)
print("tensorflowjs", getattr(tfjs, "__version__", "ok"))

## 3. Assert required inputs exist

In [ ]:
from pathlib import Path

DECAFIA_ZIP = Path("/content/decafia.zip")
SCRIPTS_DIR = Path("/content/training")
REQUIRED = [
    DECAFIA_ZIP,
    SCRIPTS_DIR / "prepare_decafia.py",
    SCRIPTS_DIR / "train.py",
    SCRIPTS_DIR / "export_tfjs.py",
]

missing = [str(p) for p in REQUIRED if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing required Colab files:\n  - "
        + "\n  - ".join(missing)
        + "\n\nUpload decafia.zip to /content/ and the three .py scripts to /content/training/."
    )

print("All required inputs found:")
for p in REQUIRED:
    print(" ", p, f"({p.stat().st_size} bytes)")

## 4. Unzip DECAFIA → `/content/decafia_raw`

In [ ]:
import shutil
from pathlib import Path

DECAFIA_ZIP = Path("/content/decafia.zip")
RAW_DIR = Path("/content/decafia_raw")

if not DECAFIA_ZIP.is_file():
    raise FileNotFoundError(f"Missing {DECAFIA_ZIP}")

if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

!unzip -q /content/decafia.zip -d /content/decafia_raw

files = [p for p in RAW_DIR.rglob("*") if p.is_file() and "__MACOSX" not in str(p)]
if not files:
    raise FileNotFoundError(f"{RAW_DIR} is empty after unzip. Is decafia.zip valid?")

print(f"Unzipped OK — {len(files)} files under {RAW_DIR}")
!find /content/decafia_raw -maxdepth 3 -type d | head -40

## 5. Remap YOLO → classification folders

`healthy` / `pest_like` / `disease_like` → `/content/decafia_remapped`

In [ ]:
from pathlib import Path

PREPARE = Path("/content/training/prepare_decafia.py")
RAW_DIR = Path("/content/decafia_raw")
REMAPPED = Path("/content/decafia_remapped")

if not PREPARE.is_file():
    raise FileNotFoundError(f"Missing {PREPARE}")
if not RAW_DIR.exists():
    raise FileNotFoundError(f"Missing {RAW_DIR} — run the unzip cell first.")

!python /content/training/prepare_decafia.py --source /content/decafia_raw --out /content/decafia_remapped

classes = ["healthy", "pest_like", "disease_like"]
exts = {".jpg", ".jpeg", ".png"}
counts = {}
for split in ("train", "val"):
    for cls in classes:
        d = REMAPPED / split / cls
        if not d.is_dir():
            raise FileNotFoundError(f"Expected remapped folder missing: {d}")
        n = sum(1 for p in d.iterdir() if p.suffix.lower() in exts)
        counts[f"{split}/{cls}"] = n

total = sum(counts.values())
if total == 0:
    raise FileNotFoundError("Remap produced 0 images.")

print("Remap counts:")
for k, v in counts.items():
    print(f"  {k}: {v}")
print("Total images:", total)

## 6. Train MobileNetV3Small

Writes `/content/teman_kopi_keras.keras`

In [ ]:
from pathlib import Path

TRAIN = Path("/content/training/train.py")
DATA = Path("/content/decafia_remapped")
KERAS_OUT = Path("/content/teman_kopi_keras.keras")

if not TRAIN.is_file():
    raise FileNotFoundError(f"Missing {TRAIN}")
if not (DATA / "train").is_dir():
    raise FileNotFoundError(f"Missing {DATA / 'train'} — run remap cell first.")

!python /content/training/train.py --data-dir /content/decafia_remapped --epochs 8 --out /content/teman_kopi_keras.keras

if not KERAS_OUT.is_file():
    raise FileNotFoundError(f"Training finished but model not found: {KERAS_OUT}")
print("Keras model saved:", KERAS_OUT, KERAS_OUT.stat().st_size, "bytes")

## 7. Export TensorFlow.js → `/content/teman-kopi-tfjs`

In [ ]:
from pathlib import Path
import shutil

EXPORT = Path("/content/training/export_tfjs.py")
KERAS_OUT = Path("/content/teman_kopi_keras.keras")
TFJS_OUT = Path("/content/teman-kopi-tfjs")

if not EXPORT.is_file():
    raise FileNotFoundError(f"Missing {EXPORT}")
if not KERAS_OUT.is_file():
    raise FileNotFoundError(f"Missing {KERAS_OUT} — run train cell first.")

if TFJS_OUT.exists():
    shutil.rmtree(TFJS_OUT)

!python /content/training/export_tfjs.py --model /content/teman_kopi_keras.keras --out /content/teman-kopi-tfjs

model_json = TFJS_OUT / "model.json"
if not model_json.is_file():
    raise FileNotFoundError(f"TF.js export missing model.json at {model_json}")

print("TF.js export OK:")
for p in sorted(TFJS_OUT.iterdir()):
    print(" ", p.name, p.stat().st_size, "bytes")

## 8. Zip for download → PWA

1. Download `/content/teman-kopi-tfjs.zip`  
2. Extract into repo `public/models/teman-kopi/`  
3. Locally: `npm run build && npx vercel --prod --yes`

In [ ]:
from pathlib import Path

TFJS_OUT = Path("/content/teman-kopi-tfjs")
ZIP_PATH = Path("/content/teman-kopi-tfjs.zip")

if not (TFJS_OUT / "model.json").is_file():
    raise FileNotFoundError(
        f"Missing {TFJS_OUT / 'model.json'} — run export cell first."
    )

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

!cd /content && zip -qr teman-kopi-tfjs.zip teman-kopi-tfjs

if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Failed to create {ZIP_PATH}")

print("Download this file from Colab Files:")
print(" ", ZIP_PATH, f"({ZIP_PATH.stat().st_size} bytes)")
print()
print("Then copy extracted files into:")
print("  temankopi/public/models/teman-kopi/")
print("and run: npm run build && npx vercel --prod --yes")